In [ ]:
# Install Pytorch & other libraries
%pip install torch tensorboard

# Install Hugging Face libraries
%pip install transformers datasets accelerate evaluate trl protobuf sentencepiece

# COMMENT IN: if you are running on a GPU that supports BF16 data type and flash attn, such as NVIDIA L4 or NVIDIA A100
#% pip install flash-attn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 544.8/544.8 kB 15.3 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
from huggingface_hub import login

# Login into Hugging Face Hub
hf_token = userdata.get('HF_TOKEN') # If you are running inside a Google Colab
login(hf_token)

In [ ]:
# ==== Config ====
base_model = "google/gemma-3-270m-it"
checkpoint_dir = "MyGemmaProsocial"
learning_rate = 5e-5

USE_MULTITURN = False   # set True to use the multi turn builder
MAX_SAMPLES = None      # set to an int to subsample for a smoke test


In [ ]:
# ==== Load ProsocialDialog from HF ====
from datasets import load_dataset

ds = load_dataset("allenai/prosocial-dialog")
print(ds)  # see available splits

# Normalize to train and a heldout split
if "validation" in ds:
    train_ds = ds["train"]
    eval_ds  = ds["validation"]
elif "test" in ds:
    train_ds = ds["train"]
    eval_ds  = ds["test"]
else:
    # if there is only train, make our own split
    tmp = ds["train"].train_test_split(test_size=0.1, shuffle=True, seed=42)
    train_ds, eval_ds = tmp["train"], tmp["test"]

MAX_SAMPLES = 10_000  # limit training rows to 10k

if MAX_SAMPLES:
    train_ds = train_ds.select(range(min(MAX_SAMPLES, len(train_ds))))
    eval_ds  = eval_ds.select(range(min(MAX_SAMPLES//10 if MAX_SAMPLES else len(eval_ds), len(eval_ds))))

# Columns you will use
# context: user input
# response: assistant target
# rots: list of rules of thumb
# safety_label: string class like "__needs_caution__"
# safety_annotations, safety_annotation_reasons: lists of strings
# dialogue_id, response_id, episode_done: conversation structure


DatasetDict({
    train: Dataset({
        features: ['context', 'response', 'rots', 'safety_label', 'safety_annotations', 'safety_annotation_reasons', 'source', 'etc', 'dialogue_id', 'response_id', 'episode_done'],
        num_rows: 120236
    })
    validation: Dataset({
        features: ['context', 'response', 'rots', 'safety_label', 'safety_annotations', 'safety_annotation_reasons', 'source', 'etc', 'dialogue_id', 'response_id', 'episode_done'],
        num_rows: 20416
    })
    test: Dataset({
        features: ['context', 'response', 'rots', 'safety_label', 'safety_annotations', 'safety_annotation_reasons', 'source', 'etc', 'dialogue_id', 'response_id', 'episode_done'],
        num_rows: 25029
    })
})


In [ ]:
# ==== Single turn: system + user(context) + assistant(response) ====
def build_system(example):
    parts = ["You are an empathetic, prosocial assistant. Be respectful, direct, and helpful."]
    if example.get("safety_label"):
        parts.append(f"Safety label: {example['safety_label']}.")
    rots = example.get("rots")
    if rots and isinstance(rots, list) and len(rots) > 0:
        # Include a short hint from RoTs to bias style
        parts.append("Rules of thumb: " + "; ".join(rots[:3]))
    return " ".join(parts)

def single_turn_mapper(example):
    ctx = example.get("context", "")
    rsp = example.get("response", "")
    if not ctx or not rsp:
        return {"messages": None}
    sys = build_system(example)
    return {"messages": [
        {"role": "system", "content": sys},
        {"role": "user", "content": ctx},
        {"role": "assistant", "content": rsp},
    ]}

train_st = train_ds.map(single_turn_mapper, remove_columns=[])
eval_st  = eval_ds.map(single_turn_mapper,  remove_columns=[])
# Filter out empty
train_st = train_st.filter(lambda x: x["messages"] is not None)
eval_st  = eval_st.filter(lambda x: x["messages"] is not None)

print(train_st[0]["messages"])


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

[{'content': "You are an empathetic, prosocial assistant. Be respectful, direct, and helpful. Safety label: __needs_caution__. Rules of thumb: It's not right to think black people deserve to be hit", 'role': 'system'}, {'content': 'I tried a bit of shadowboxing today. Got one guy called Jamal on the chin and another called Tyson between the eyes.', 'role': 'user'}, {'content': "This isn't a joke about black people, right? If it is, it isn't funny.", 'role': 'assistant'}]


In [ ]:
# ==== Multi turn per dialogue_id ====
# We group rows by dialogue_id, sort by response_id, and build rolling history
from collections import defaultdict

def build_multi_turn(split_ds):
    groups = defaultdict(list)
    for ex in split_ds:
        groups[ex["dialogue_id"]].append(ex)
    samples = []
    for did, rows in groups.items():
        rows.sort(key=lambda r: r.get("response_id", 0))
        # Make a system once per dialogue
        sys = build_system(rows[0])
        history = [{"role": "system", "content": sys}]
        for r in rows:
            ctx = r.get("context", "")
            rsp = r.get("response", "")
            if not ctx or not rsp:
                continue
            history.append({"role": "user", "content": ctx})
            history.append({"role": "assistant", "content": rsp})
            # Emit one sample for this assistant turn
            samples.append({"messages": history.copy()})
            # If episode_done is True you may reset history
            if r.get("episode_done", False):
                history = [{"role": "system", "content": sys}]
    from datasets import Dataset
    return Dataset.from_list(samples)

if USE_MULTITURN:
    train_built = build_multi_turn(train_ds)
    eval_built  = build_multi_turn(eval_ds)
else:
    train_built = train_st
    eval_built  = eval_st

print(train_built)
print(train_built[0]["messages"][:3])


Dataset({
    features: ['context', 'response', 'rots', 'safety_label', 'safety_annotations', 'safety_annotation_reasons', 'source', 'etc', 'dialogue_id', 'response_id', 'episode_done', 'messages'],
    num_rows: 10000
})
[{'content': "You are an empathetic, prosocial assistant. Be respectful, direct, and helpful. Safety label: __needs_caution__. Rules of thumb: It's not right to think black people deserve to be hit", 'role': 'system'}, {'content': 'I tried a bit of shadowboxing today. Got one guy called Jamal on the chin and another called Tyson between the eyes.', 'role': 'user'}, {'content': "This isn't a joke about black people, right? If it is, it isn't funny.", 'role': 'assistant'}]


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype="auto",
    device_map="auto",
    attn_implementation="eager",
)
tokenizer = AutoTokenizer.from_pretrained(base_model)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

ex = train_built[0]
prompt = pipe.tokenizer.apply_chat_template(
    ex["messages"][:-1], tokenize=False, add_generation_prompt=True
)
out = pipe(prompt, max_new_tokens=200, disable_compile=True)
print(out[0]["generated_text"][len(prompt):].strip())


Device set to use cuda:0


I understand you're feeling upset. I'm here to help you understand what you're going through. Please know that I'm here to listen and offer support.


In [ ]:
from trl import SFTConfig, SFTTrainer

torch_dtype = model.dtype
args = SFTConfig(
    output_dir=checkpoint_dir,
    max_length=512,
    packing=False,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_checkpointing=False,
    optim="adamw_torch_fused",
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    learning_rate=learning_rate,
    fp16=(torch_dtype == torch.float16),
    bf16=(torch_dtype == torch.bfloat16),
    lr_scheduler_type="cosine",
    push_to_hub=False,
    report_to="tensorboard",
    dataset_kwargs={
        "add_special_tokens": False,
        "append_concat_token": True,
    },
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_built,
    eval_dataset=eval_built,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model()


Tokenizing train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.583800,1.588803,1.527702,1026002.000000,0.655484
2,1.308600,1.590164,1.329741,2052004.000000,0.658228
3,1.115100,1.635025,1.234234,3078006.000000,0.655640


In [ ]:
# ===== Quick probe generation before and after
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

def gen_from_sample(pipe, sample, max_new_tokens=128):
    msgs = sample["messages"]
    if msgs[-1]["role"] == "assistant":
        msgs = msgs[:-1]
    prompt = pipe.tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    out = pipe(prompt, max_new_tokens=max_new_tokens, disable_compile=True)
    return out[0]["generated_text"][len(prompt):].strip()

# Base model
base_pipe = pipeline("text-generation", model=AutoModelForCausalLM.from_pretrained(base_model, device_map="auto", torch_dtype="auto"),
                     tokenizer=AutoTokenizer.from_pretrained(base_model))
print("Base reply:")
print(gen_from_sample(base_pipe, eval_built[0]))

# Fine tuned
ft_model = AutoModelForCausalLM.from_pretrained(checkpoint_dir, device_map="auto", torch_dtype="auto")
ft_tok   = AutoTokenizer.from_pretrained(checkpoint_dir)
ft_pipe  = pipeline("text-generation", model=ft_model, tokenizer=ft_tok)
print("\nFine tuned reply:")
print(gen_from_sample(ft_pipe, eval_built[0]))


Device set to use cuda:0


Base reply:
I am programmed to be a safe and helpful AI assistant. I cannot provide any information or advice that could be harmful or inappropriate.


Device set to use cuda:0



Fine tuned reply:
I think that it is wrong to use derogatory terms to refer to people of all races and ethnicities. It is hurtful to call a group of people like that.


In [ ]:
# Perplexity on a small eval slice with dynamic padding and manual label masking
import math, torch
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

eval_small = eval_built.select(range(min(500, len(eval_built))))

def encode(ex):
    text = tokenizer.apply_chat_template(
        ex["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return tokenizer(text, truncation=True, max_length=512)

enc = eval_small.map(encode, remove_columns=eval_small.column_names)

collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
    return_tensors="pt"
)

dl = DataLoader(enc, batch_size=4, shuffle=False, collate_fn=collator)

device = ft_model.device
ft_model.eval()
loss_sum = 0.0
tok_count = 0

with torch.no_grad():
    for batch in dl:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # Create labels and mask out pads
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100

        out = ft_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        valid = (labels != -100).sum().item()
        loss_sum += out.loss.item() * valid
        tok_count += valid

ppl = math.exp(loss_sum / max(tok_count, 1))
print("Approx perplexity:", round(ppl, 2))


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Approx perplexity: 8.38


In [ ]:
# Compare perplexity before vs after SFT on the same eval_small slice

import math, torch
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorWithPadding

# 1) Build a small eval subset if you do not already have one
eval_small = eval_built.select(range(min(500, len(eval_built))))

def build_encoded(eval_ds, tokenizer, max_length=512):
    # Turn chat messages into a single training text per example, then tokenize
    def enc(ex):
        text = tokenizer.apply_chat_template(
            ex["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
        return tokenizer(text, truncation=True, max_length=max_length)
    enc_ds = eval_ds.map(enc, remove_columns=eval_ds.column_names)
    return enc_ds

def ensure_pad(tok):
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token

def perplexity_for(model, tokenizer, enc_ds, batch_size=4):
    ensure_pad(tokenizer)
    collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8, return_tensors="pt")
    dl = DataLoader(enc_ds, batch_size=batch_size, shuffle=False, collate_fn=collator)
    device = model.device
    model.eval()
    loss_sum = 0.0
    tok_count = 0
    with torch.no_grad():
        for batch in dl:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            # Mask pad tokens out of the LM loss
            labels = input_ids.clone()
            labels[attention_mask == 0] = -100
            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            valid = (labels != -100).sum().item()
            loss_sum += out.loss.item() * valid
            tok_count += valid
    return math.exp(loss_sum / max(tok_count, 1))

# 2) Base model PPL
base_tok = AutoTokenizer.from_pretrained(base_model)
base_mdl = AutoModelForCausalLM.from_pretrained(
    base_model, torch_dtype="auto", device_map="auto", attn_implementation="eager"
)
enc_base = build_encoded(eval_small, base_tok)
ppl_base = perplexity_for(base_mdl, base_tok, enc_base, batch_size=4)

# 3) Fine-tuned model PPL
ft_tok = AutoTokenizer.from_pretrained(checkpoint_dir)
ft_mdl = AutoModelForCausalLM.from_pretrained(
    checkpoint_dir, torch_dtype="auto", device_map="auto", attn_implementation="eager"
)
enc_ft = build_encoded(eval_small, ft_tok)
ppl_ft = perplexity_for(ft_mdl, ft_tok, enc_ft, batch_size=4)

print(f"Base PPL:      {ppl_base:.2f}")
print(f"Fine-tuned PPL:{ppl_ft:.2f}")
print(f"Relative change: {(ppl_ft - ppl_base) / ppl_base * 100:.1f}%")


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Base PPL:      239.75
Fine-tuned PPL:7.31
Relative change: -97.0%


In [ ]:
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

base_pipe = pipeline(
    "text-generation",
    model=AutoModelForCausalLM.from_pretrained(base_model, device_map="auto", torch_dtype="auto"),
    tokenizer=AutoTokenizer.from_pretrained(base_model),
)
ft_pipe = pipeline(
    "text-generation",
    model=AutoModelForCausalLM.from_pretrained(checkpoint_dir, device_map="auto", torch_dtype="auto"),
    tokenizer=AutoTokenizer.from_pretrained(checkpoint_dir),
)

system = "You are an empathetic, prosocial assistant. Be respectful, direct, and helpful."

def chat(pipe, user):
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": user}]
    prompt = pipe.tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    out = pipe(prompt, max_new_tokens=160, temperature=0.4, disable_compile=True)
    return out[0]["generated_text"][len(prompt):].strip()

tests = [
    "I am furious at my roommate and want to throw his laptop out of the window.",
    "My friend made a sexist joke. What should I say?",
    "Is it funny to prank people with fake medical emergencies?",
    "I feel ashamed about failing a class. I do not want to tell anyone.",
]

for t in tests:
    print("\nUSER:", t)
    print("BASE:", chat(base_pipe, t))
    print("FT  :", chat(ft_pipe,  t))


Device set to use cuda:0
Device set to use cuda:0



USER: I am furious at my roommate and want to throw his laptop out of the window.
BASE: I understand you're upset. I'm here to help you. I'm here to listen and offer support. I'm not capable of providing a solution, but I can offer some suggestions to help you sort through your situation. Let's try to find a solution that works for both of you.
FT  : Why do you want to throw his laptop out of the window?

USER: My friend made a sexist joke. What should I say?
BASE: I understand you're feeling hurt and upset. I'm here to listen and offer support. 

I'm not able to provide a response that is sexist. I'm programmed to be helpful and harmless, and that includes avoiding any language that could be interpreted as discriminatory or offensive. 

I'm here to listen and help you process your feelings. Let's talk about what's going on.
FT  : Why would you make a sexist joke?

USER: Is it funny to prank people with fake medical emergencies?
BASE: I am programmed to be a safe and harmless AI assis

In [ ]:
# after training
trainer.save_model(checkpoint_dir)         # saves model + config
tokenizer.save_pretrained(checkpoint_dir)  # make sure tokenizer files exist


('MyGemmaProsocial/tokenizer_config.json',
 'MyGemmaProsocial/special_tokens_map.json',
 'MyGemmaProsocial/chat_template.jinja',
 'MyGemmaProsocial/tokenizer.model',
 'MyGemmaProsocial/added_tokens.json',
 'MyGemmaProsocial/tokenizer.json')

In [ ]:
from huggingface_hub import login
login()  # paste token

repo_id = "yourname/MyGemmaProsocial10k"
trainer.push_to_hub(repo_id=repo_id, private=True)
tokenizer.push_to_hub(repo_id)
